In [1]:
from pathlib import Path
import sys

helper_dir = Path.cwd() / "HelperCode"
sys.path.insert(0, str(helper_dir))


from notebook_setup import *

## Hard-Coded Parameters

| Parameter | Description |
| :--- | :--- |
| `round_tol` | Numerical tolerance used by `nuc_wavefxn.get_ssct_parameters`; numerical errors smaller than this value are suppressed. |
| `nuc_q_eps` | Nuclear state-preparation error from unitary synthesis allocation, $\epsilon^{(n,q)} = \sum_i \epsilon^{(n,q)}_{i0}$. |
| `elc_q_eps` | Electronic state-preparation error from unitary synthesis allocation, $\epsilon^{(e,q)} = \sum_a \epsilon^{(e,q)}_a$. |
| `N_SMB` | Number of single-mode basis states, $N_{\mathrm{SMB}}$. |
| `b_grad` | Number of qubits used to represent the phase-gradient state. |
| `b_rot_elec` | Number of precision qubits used to synthesize the rotations in the electronic MPS preparation circuit. |
| `nuc_error_weight` | Controls the allocation of error between $\epsilon^{(n,c)} = \sum_i \epsilon^{(n,c)}_{i0}$ and $\epsilon_{\mathrm{shear}}$. |
| `qubit_search_low_bound` | Lower bound of the qubit range searched by `grid_parameters.get_nuc_grid_parameters`. |
| `qubit_search_high_bound` | Upper bound of the qubit range searched by `grid_parameters.get_nuc_grid_parameters`. |
| `L_search_range` | Sets the upper end of the length search interval to `L_search_range * L_elec`. The interval begins at `L_elec`. |
| `L_search_step_size` | Step size between successive lengths examined by `grid_parameters.get_nuc_grid_parameters`. |

In [2]:
round_tol = 1e-14
nuc_q_eps = 1e-12
elc_q_eps = 1e-12

N_SMB = 1
b_grad = 64 
b_rot_elec = 70

nuc_error_weight = 1.0/100.0

qubit_search_low_bound = 13
qubit_search_high_bound = 18

L_search_range = 100
L_search_step_size = 100

## Molecular Library

Set the variable `molecule`, defined below, to the name of the MolecularData directory of the molecule that resource esimates will be computed for. For example, `molecule = "H2"`.

To compute resources for a molecule in Table 4 of [1], set `molecule` to one of the options in `molecule_library`.

| Variable | Description |
| :--- | :--- |
| `molecule_library` | Array containing the names of the molecules listed in Table 4. |
| `molecule` | Molecule for which the resource estimates will be calculated. |

Some directory names differ from the corresponding molecular names used in Table 4:

| Directory name | Name in Table 4 |
| :--- | :--- |
| `Criegee` | $\mathrm{CH_2OO}$ |
| `Cyclobutanone` | $\mathrm{C_4H_6O}$ |
| `HPALD` | $\mathrm{C_5\text{-}HPALD}$ |
| `HCFC` | $\mathrm{HCFC\text{-}132b}$ |

In [29]:
#molecule = "H2"

molecule_library = np.array(["CH4", "Criegee",
                             "Cyclobutanone", 
                             "CF3CO2H", "HPALD",  
                             "HCFC", "CH3OBr",
                             "BrCH2CHO"])

molecule = molecule_library[4] #select a molecule from molecular library

## File Setup

Locates the input files required for the calculation and defines the corresponding file-path variables.

In [30]:
parameter_extn = isp_helper_functions.get_param_file_extn(molecule)
psi4_extn = isp_helper_functions.get_psi4_file_extn(molecule)
elec_mps_extn = isp_helper_functions.get_elec_mps_file_extn(molecule)
output_extn = isp_helper_functions.get_output_file_extn(molecule)
error_budget_extn = isp_helper_functions.get_error_budget_extn(molecule)
lambda_mat_extn = isp_helper_functions.get_error_lambda_mat_extn(molecule)

parameter_file = root_dir / parameter_extn.lstrip("/")
psi4_out_file = root_dir / psi4_extn.lstrip("/")
elec_mps_file = root_dir / elec_mps_extn.lstrip("/")
output_data_file = root_dir / output_extn.lstrip("/")
error_budget_file = root_dir / error_budget_extn.lstrip("/")
lambda_mat_file = root_dir / lambda_mat_extn.lstrip("/")

## Read-in Parameters

Reads the input parameters required by subsequent calculations.

In [31]:
parameters = io.read_file(parameter_file)
elec_parameters = io.read_file(elec_mps_file)
error_budget = io.read_file(error_budget_file)

## Vibrational Hartree-Product Calculation

The following code extracts the molecular Hessian and atomic masses from the Psi4 output, constructs the mass-weighting matrix, and checks the resulting vibrational frequencies.

If successful, the code will output the vibrational frequencies found by diagonalizing the Hessian . For example, when `molecule = H2`:

```
Frequencies in cm^-1: [4501.41357453] 
 ```

These frequencies should be close to the vibrational frequencies in the `psi4.out` file. If not, something went wrong.


| Variable | Description |
| :--- | :--- |
| `eta_n` | Number of nuclei in `molecule`, denoted by $\eta_n$ on page 3. |
| `is_linear` | Boolean indicating whether `molecule` is linear. |
| `pre_weighted_hessian` | Molecular Hessian obtained from the Psi4 calculation before mass weighting is applied. |
| `molec_masses` | Dictionary mapping each atomic index to the mass of the corresponding atom in `molecule`. |
| `mass_weight_mat` | Matrix used to apply mass weighting to the molecular Hessian. |
| `freqs` | Vibrational frequencies obtained from the molecular Hessian. |

In [32]:
eta_n = parameters["nuclei"]
is_linear = parameters["is_linear"]

pre_weighted_hessian = parse_psi4.get_hessian(psi4_out_file, eta_n)
_, molec_masses, _ = parse_psi4.get_molec_data(psi4_out_file, eta_n)

mass_weight_mat, _ = nuc_wavefxn.get_mass_weight_mat(molec_masses, eta_n)
_, freqs = nuc_wavefxn.check_freqs(molec_masses, eta_n, pre_weighted_hessian, is_linear)

Frequencies in cm^-1: [   4.07646314   43.3442544    90.16620164  112.46228291  146.46008428
  170.94698973  175.97724629  239.65977489  294.88279177  350.25955524
  377.48008355  438.66111573  542.86708994  638.38786115  821.88310064
  857.27864322  907.04913125  996.25183137 1013.76349524 1040.90990582
 1065.94099022 1088.56987004 1136.75201749 1224.02314265 1267.80344454
 1348.38699485 1364.28199903 1396.71114505 1412.94896873 1433.6004196
 1471.54361167 1482.88341833 1512.62952144 1713.48542461 1746.92311441
 3008.27559426 3071.06722467 3093.42620746 3144.55545503 3184.69903435
 3197.00652892 3207.37031288 3788.89839538] 



## SSCT Parameters

The following code block computes diagonal matrix `Sigma` and right triangular matrix `R_mat` such that `R_mat.transpose` $\times$ `Sigma` $\times$ `R_mat` is a positive definite matrix that defines the initial Gaussian state

In [33]:
Sigma, R_mat = nuc_wavefxn.get_ssct_parameters(molec_masses, eta_n, pre_weighted_hessian, 
                                               round_tol, is_linear)

Lambda = np.matmul(np.transpose(R_mat), np.matmul(Sigma, R_mat))

## Compute the Remaining ISP Error Budget

After accounting for the fixed error contributions, the remaining initial-state-preparation (ISP) error budget is allocated between preparation of the nuclear state and the normal-coordinate transformation. The next code block computes the remaining error.

The nuclear and electronic rotation-synthesis errors, $\epsilon^{(n,q)}$ and $\epsilon^{(e,q)}$, have been hardcoded to sufficiently small values that their contributions to the total ISP error is minimal.

The remaining budget is

$$
\epsilon_{\mathrm{ISP}}^{(\mathrm{remaining})}
=
\epsilon_{\mathrm{ISP}}
-
\left(
\epsilon_{\mathrm{PKB}}
+
\epsilon_{\mathrm{trim}}
+
\epsilon^{(e,c)}
+
\epsilon^{(n,q)}
+
\epsilon^{(e,q)}
\right).
$$

| Variable | Description |
| :--- | :--- |
| `total_isp_budget` | Total error budget allocated to initial-state preparation. |
| `pkb_budget` | Error budget allocated to phase kickback. |
| `trim_budget` | Error budget allocated to grid trimming. |
| `elec_c_eps` | Electronic MPS preparation error, read from the electronic MPS results file. |
| `nuc_q_eps` | Nuclear rotation-synthesis error allocation, $\epsilon^{(n,q)}$. |
| `elc_q_eps` | Electronic rotation-synthesis error allocation, $\epsilon^{(e,q)}$. |
| `remaining_ISP_budget` | ISP error budget remaining for nuclear-state preparation and the normal-coordinate transformation. |

The value of `elec_c_eps` is read from the electronic MPS results file because it is computed during the electronic MPS calculation.

In [34]:
total_isp_budget = error_budget["isp_budget"]
pkb_budget = error_budget["pkb_budget"]
trim_budget = error_budget["trim_budget"]
elec_c_eps = elec_parameters["elec_isp_eps"] 

remaining_ISP_budget = total_isp_budget - (pkb_budget + trim_budget + elec_c_eps + 
                                           nuc_q_eps + elc_q_eps)

## Compute the Nuclear Grid Length and Momentum Cutoff

Next, the length of the nuclear grid and the nuclear momentum cutoff are calculated. To do so, the remaining ISP error budget is divided between the nuclear momentum-cutoff error and the single-shear coordinate-transformation (SSCT) error:

$$
\epsilon^{(n,c)}
=
w\,
\epsilon_{\mathrm{ISP}}^{(\mathrm{remaining})},
$$

$$
\epsilon_{\mathrm{SSCT}}
=
    \left(1-w\right)
\epsilon_{\mathrm{ISP}}^{(\mathrm{remaining})},
$$
where $w = $ `nuc_error_weight`.

The nuclear grid search then selects a qubit count, grid length, and momentum cutoff satisfying the supplied error and grid constraints.

| Variable | Description |
| :--- | :--- |
| `E_elec` | Electronic kinetic energy found from electronic MPS calculation. |
| `L_elec` | Simulation-cell length found from electronic MPS calculation. |
| `nuc_c_budget` | Error budget allocated to erro cause from the nuclear momentum cutoff, $\epsilon^{(n,c)}$. |
| `ssct_budget` | Error budget allocated to the SSCT, $\epsilon_{\mathrm{SSCT}}$. |
| `K_max_elec` | Electronic momentum cutoff, $K_{\max}^{(e)}$. |
| `Sigma` | Diagonal matrix containing the nuclear Gaussian coefficients. |
| `n_isp` | Number of qubits per coordinate that causes error budget to be satisfied. |
| `L_nuc` | Nuclear grid length that that causes error budget to be satisfied. |
| `K_max_nuc` |  Nuclear momentum cutoff that that causes error budget to be satisfied.|

In [35]:
E_elec = elec_parameters["E_elec"]
L_elec = elec_parameters["L_elec"]

nuc_c_budget = nuc_error_weight * remaining_ISP_budget
ssct_budget =  (1 - nuc_error_weight) * remaining_ISP_budget

L_search_low = L_elec
L_search_high = L_search_range * L_search_low

K_max_elec = np.sqrt(2 * E_elec) 


n_isp, L_nuc, K_max_nuc = grid_parameters.get_nuc_grid_parameters(Sigma, eta_n, 
                                                                  qubit_search_low_bound,
                                                                  qubit_search_high_bound, 
                                                                  L_search_low, 
                                                                  L_search_high, 
                                                                  ssct_budget, nuc_c_budget, 
                                                                  K_max_elec, L_elec, 
                                                                  L_search_step_size)

## Compute the Padded Nuclear Grid

The following code block computes the number of padding qubits that are needed to prevent modular wraparound during the transformation.

The padded register size is

$$
\bar{n}_{\mathrm{ISP}}
=
n_{\mathrm{ISP}} + n_{\mathrm{pad}}.
$$

| Variable | Description |
| :--- | :--- |
| `n_isp` | Number of qubits per nuclear coordinate before padding. |
| `R_mat` | Linear transformation matrix used to determine the required padding. |
| `n_pad` | Number of additional qubits required per nuclear coordinate. |
| `n_isp_bar` | Total number of qubits per nuclear coordinate after padding. |

In [36]:
n_pad = grid_parameters.get_padded_grid_params(n_isp, R_mat)
n_isp_bar = n_isp + n_pad

## Estimate Nuclear ISP Resources

The nuclear initial-state-preparation resources are estimated in four steps:

1. Convert the diagonal Gaussian coefficients in `Sigma` to their position-space representation.
2. Determine the maximum nuclear MPS bond dimension.
3. Determine the rotation-synthesis precision required to prepare the nuclear MPS.
4. Estimate the ancilla, data-qubit, and Toffoli costs of each component of the nuclear ISP procedure.

The position-space Gaussian coefficients are computed elementwise as

$$
\mathtt{Sigma}\_\mathtt{ps}[i] = \frac{1}{\Sigma_{ii}}.
$$

| Variable | Description |
| :--- | :--- |
| `Sigma_ps` | One-dimensional array containing the position-space Gaussian coefficients. |
| `nuc_bond_dim_max` | Maximum bond dimension required for the nuclear MPS. |
| `b_rot_nuc` | Number of precision qubits used to synthesize rotations during nuclear MPS preparation. |
| `W_nuc_anc` | Ancilla-qubit cost of preparing the nuclear MPS. |
| `W_nuc_Toff` | Toffoli-gate cost of preparing the nuclear MPS. |
| `U_A_inv_trans_anc` | Ancilla-qubit cost of applying the inverse-transpose coordinate transformation, $\mathbf{A}^{-\top}$. |
| `U_A_inv_trans_Toff` | Toffoli-gate cost of applying the inverse-transpose coordinate transformation, $\mathbf{A}^{-\top}$. |
| `PK_anc` | Ancilla-qubit cost of phase kickback. |
| `PK_Toff` | Toffoli-gate cost of phase kickback. |
| `nuc_isp_Toff` | Total Toffoli-gate cost of nuclear initial-state preparation. |
| `nuc_isp_anc` | Peak ancilla-qubit cost of nuclear initial-state preparation. |
| `nuc_data_qubits` | Number of qubits used to store the nuclear wavefunction. |

In [37]:
Sigma_ps = 1.0/np.diag(Sigma) #Gaussian coefficients in position space

#Compute maximum bond dimension
nuc_bond_dim_max = int(isp_resources.get_nuc_bond_dim_max(nuc_c_budget, Sigma_ps, L_nuc, N_SMB))

#Compute rotational qubits for nuclear unitary synthesis
b_rot_nuc = isp_resources.get_b_rot_nuc(nuc_q_eps, nuc_bond_dim_max, n_isp, eta_n)


(
    W_nuc_anc, 
    W_nuc_Toff, 
    U_A_inv_trans_anc, 
    U_A_inv_trans_Toff, 
    PK_anc, 
    PK_Toff, 
    nuc_isp_Toff, 
    nuc_isp_anc,
    nuc_data_qubits
) = isp_resources.get_nuc_resources(N_SMB, 
                                    n_isp, 
                                    n_isp_bar, 
                                    b_rot_nuc, 
                                    b_grad, 
                                    nuc_bond_dim_max, 
                                    eta_n, 
                                    pkb_budget)

## Estimate Electronic ISP Resources

The following code block computes the resources used to prepare the  electronic initial state.

| Variable | Description |
| :--- | :--- |
| `eta_e` | Number of electrons in the molecule, $\eta_e$. |
| `elec_bond_dim_max` | Maximum bond dimension encountered during electronic MPS preparation. |
| `N_MOB` | Number of molecular-orbital basis functions, $N_{\mathrm{MOB}}$. |
| `ASYM_anc` | Ancilla-qubit cost of antisymmetrizing the electronic state. |
| `ASYM_toff` | Toffoli-gate cost of antisymmetrizing the electronic state. |
| `W_elec_anc` | Ancilla-qubit cost of preparing the electronic MPS. |
| `W_elec_toff` | Toffoli-gate cost of preparing the electronic MPS. |
| `elec_isp_Toff` | Total Toffoli-gate cost of electronic initial-state preparation. |
| `elec_isp_anc` | Peak ancilla-qubit cost of electronic initial-state preparation. |
| `elec_data_qubits` | Number of qubits used to represent the electronic spatial and spin degrees of freedom. |

In [38]:
eta_e = parameters["electrons"]
elec_bond_dim_max = elec_parameters["elec_bond_dim_max"]

N_MOB = int(np.ceil(eta_e/2.0) + 1) #number of molecular orbitals

(
    ASYM_anc,
    ASYM_toff,
    W_elec_anc,
    W_elec_toff,
    elec_isp_Toff,
    elec_isp_anc,
    elec_data_qubits,
) = isp_resources.get_elec_resources(n_isp, eta_e, N_MOB, b_rot_elec, elec_bond_dim_max)

## Compute Total ISP Resources

The final step is to compute the total initial-state-preparation resource estimates.


Because the electronic and nuclear preparation procedures are performed sequentially, their ancilla registers can be reused. The total ancilla cost is therefore the larger of the two component costs:

$$
n_{\mathrm{anc}}
=
\max\left(
n_{\mathrm{anc}}^{(e)},
n_{\mathrm{anc}}^{(n)}
\right).
$$

| Variable | Description |
| :--- | :--- |
| `isp_Toff` | Total Toffoli-gate cost of initial-state preparation. |
| `isp_anc` | Peak number of ancilla qubits required during initial-state preparation. |
| `isp_data` | Total number of nuclear and electronic data qubits. |

In [39]:
(
    isp_Toff, 
    isp_anc, 
    isp_data
) = isp_resources.get_total_resources(elec_isp_Toff, 
                                      elec_isp_anc, 
                                      elec_data_qubits,
                                      nuc_isp_Toff,
                                      nuc_isp_anc,
                                      n_isp)

## Resource Message

Prints a message containing resource estimates

In [40]:
delta_L = L_nuc/np.power(2, n_isp)

isp_message = isp_report.format_isp_summary(
    L=L_nuc,
    nuc_k_max=K_max_nuc,
    n_p=n_isp,
    nuc_data_qubits=nuc_data_qubits,
    delta_L=delta_L,
    nuc_bond_dim_max=nuc_bond_dim_max,
    W_nuc_Toff=W_nuc_Toff,
    W_nuc_anc=W_nuc_anc,
    U_A_inv_trans_Toff=U_A_inv_trans_Toff,
    U_A_inv_trans_anc=U_A_inv_trans_anc,
    PK_Toff=PK_Toff,
    PK_anc=PK_anc,
    elec_k_max=K_max_elec,
    elec_data_qubits=elec_data_qubits,
    ASYM_toff=ASYM_toff,
    ASYM_anc=ASYM_anc,
    elec_bond_dim_max=elec_bond_dim_max,
    W_elec_toff=W_elec_toff,
    W_elec_anc=W_elec_anc,
    isp_toff=isp_Toff,
    isp_anc=isp_anc,
    isp_data=isp_data
)

print(isp_message)

Nuclear Initial State Preparation
---------------------------------
Length              : 5.35e+02
Nuclear K_max       : 96.18
n_p                 : 14
Nuclear data qubits : 672
Delta L (L/N)       : 0.0327
Nuclear bond dim    : 667
W_n Toff            : 1.79e+10
W_n anc             : 620
U_A^{-T} Toff       : 4.5e+06
U_A^{-T} anc        : 76
PK Toff             : 1.3e+05
PK anc              : 275

Electronic Initial State Preparation
------------------------------------
Electronic K_max       : 95.17
Electronic data qubits : 2666
ASYM Toff              : 3.7e+03
ASYM anc               : 572
Electronic bond dim    : 127
W_elec Toff            : 2.28e+09
W_elec anc             : 402

Total Initial State Preparation
-------------------------------
Toffolis    : 2.02e+10
Ancilla     : 620
Data qubits : 2680


## Save Final Data

The final grid parameters and component-level ISP resource estimates are written to `output_data_file` as key-value pairs. The matrix `Lambda` is written separately to `lambda_mat_file` as a comma-separated array.

| Output entry | Description |
| :--- | :--- |
| `L` | Length of simulation grid that both nuclei and electrons live on, `L_nuc`. |
| `n_p` | Number of qubits per coordinate before padding, `n_isp`. |
| `elec_isp_Toff` | Total electronic ISP Toffoli-gate cost. |
| `elec_isp_anc` | Peak electronic ISP ancilla-qubit cost. |
| `nuc_isp_Toff` | Total nuclear ISP Toffoli-gate cost. |
| `nuc_isp_anc` | Peak nuclear ISP ancilla-qubit cost. |
| `elec_k_max` | Electronic momentum cutoff, `K_max_elec`. |
| `n_p_isp` | Number of qubits per coordinate after padding, `n_isp_bar`. |

| Output file | Contents |
| :--- | :--- |
| `output_data_file` | Scalar grid parameters and ISP resource estimates. |
| `lambda_mat_file` | Elements of `Lambda`, stored as a comma-separated matrix. | Save Final Data

In [41]:
with open(output_data_file, "w") as file:
    
    file.write("L=" + str(L_nuc) + "\n")
    file.write("n_p=" + str(n_isp) + "\n")
    file.write("elec_isp_Toff=" + str(elec_isp_Toff) + "\n")
    file.write("elec_isp_anc=" + str(elec_isp_anc) + "\n")
    file.write("nuc_isp_Toff=" + str(nuc_isp_Toff) + "\n")
    file.write("nuc_isp_anc=" + str(nuc_isp_anc) + "\n")
    file.write("elec_k_max=" + str(K_max_elec) + "\n")
    file.write("n_p_isp=" + str(n_isp_bar))
    
np.savetxt(lambda_mat_file, Lambda, delimiter=",")